# Diving in BCAM data



In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pyabf

## Exploring the `record` directory

The `records` directory contains ABF files from Joanna’s 2024 experiments. Let’s take a closer look at what these data files hold.

In [22]:
from pathlib import Path
from collections import Counter

records_dir = Path("records")
abf_files = list(records_dir.rglob("*.abf"))

print(f"The directory {records_dir} contains {len(abf_files)} ABF files.")

# Count by cell
cells = [p.parts for p in abf_files]  # p.parts gives all directories + filename
# Extract the part that contains "C1", "C2", or "C3"
cell_labels = [next((part for part in p.parts if part.startswith("C")), "Unknown") for p in abf_files]

counts = Counter(cell_labels)

print("Number of files per cel:")
for ch, n in counts.items():
    print(f"  {ch}: {n}")

The directory records contains 63 ABF files.
Number of files per cel:
  C1: 42
  C3: 16
  C2: 5


### The list of protocols

In [3]:
protocols = set()  # use a set to get unique protocols
for file in abf_files:
    abf = pyabf.ABF(file)
    protocols.add(abf.protocol)

# Convert to a sorted list
protocols_list = sorted(protocols)

print("Unique protocols found:")
for p in protocols_list:
    print("-", p)

Unique protocols found:
- IC ramp slow
- IV-DG
- IV-DG2 evry 5pA
- Slow_I_sin
- VC ramp slow 2
- gap free2


### The list of files

In [4]:
from tabulate import tabulate

# Prepare table data
table = []
for file in abf_files:
    abf = pyabf.ABF(file)
    table.append([
        str(file),
        len(abf.sweepList),
        f"{abf.sweepLengthSec:.2f}",
        abf.protocol
    ])

# Print table with headers
headers = ["File", "Nb of Sweeps", "Sweep Length (s)", "Protocol"]
print(tabulate(table, headers=headers, tablefmt="grid"))

+----------------------------------------------+----------------+--------------------+-----------------+
| File                                         |   Nb of Sweeps |   Sweep Length (s) | Protocol        |
+==============================================+================+====================+=================+
| records/2024_06/06.26/C1/2024_06_26_0025.abf |              1 |              30    | VC ramp slow 2  |
+----------------------------------------------+----------------+--------------------+-----------------+
| records/2024_06/06.26/C1/2024_06_26_0019.abf |              1 |              30    | VC ramp slow 2  |
+----------------------------------------------+----------------+--------------------+-----------------+
| records/2024_06/06.26/C1/2024_06_26_0018.abf |              1 |             100    | Slow_I_sin      |
+----------------------------------------------+----------------+--------------------+-----------------+
| records/2024_06/06.26/C1/2024_06_26_0024.abf |       

### Sorted by protocols 

In [5]:
# Sort table by "Protocol" (column index 3)
table_sorted = sorted(table, key=lambda row: row[3])

# Print table with headers
headers = ["File", "Nb of Sweeps", "Sweep Length (s)", "Protocol"]
print(tabulate(table_sorted, headers=headers, tablefmt="grid"))

+----------------------------------------------+----------------+--------------------+-----------------+
| File                                         |   Nb of Sweeps |   Sweep Length (s) | Protocol        |
+==============================================+================+====================+=================+
| records/2024_06/06.26/C1/2024_06_26_0026.abf |              1 |              30    | IC ramp slow    |
+----------------------------------------------+----------------+--------------------+-----------------+
| records/2024_06/06.26/C1/2024_06_26_0023.abf |              1 |              30    | IC ramp slow    |
+----------------------------------------------+----------------+--------------------+-----------------+
| records/2024_06/06.26/C1/2024_06_26_0020.abf |              1 |              30    | IC ramp slow    |
+----------------------------------------------+----------------+--------------------+-----------------+
| records/2024_06/06.26/C1/2024_06_26_0012.abf |       

### Sorted by the last 4 digits

In [6]:
# Sort files by the last 4 digits of the filename
abf_files_sorted = sorted(abf_files, key=lambda f: int(f.stem[-4:]))

print(f"The directory {records_dir} contains {len(abf_files_sorted)} ABF files sorted by last 4 digits:")

# Prepare table data
table = []
for file in abf_files_sorted:
    abf = pyabf.ABF(file)
    table.append([
        str(file),
        len(abf.sweepList),
        f"{abf.sweepLengthSec:.2f}",
        abf.protocol
    ])

# Print table with headers
headers = ["File", "Nb of Sweeps", "Sweep Length (s)", "Protocol"]
print(tabulate(table, headers=headers, tablefmt="github"))

The directory records contains 63 ABF files sorted by last 4 digits:
| File                                         |   Nb of Sweeps |   Sweep Length (s) | Protocol        |
|----------------------------------------------|----------------|--------------------|-----------------|
| records/2024_06/06.26/C1/2024_06_26_0000.abf |             23 |               1    | IV-DG           |
| records/2024_06/06.19/C1/2024_06_19_0000.abf |             12 |               1    | IV-DG           |
| records/2024_06/06.26/C1/2024_06_26_0001.abf |              1 |              30    | IC ramp slow    |
| records/2024_06/06.19/C1/2024_06_19_0001.abf |              1 |              30    | IC ramp slow    |
| records/2024_06/06.26/C1/2024_06_26_0002.abf |              1 |             100    | Slow_I_sin      |
| records/2024_06/06.19/C1/2024_06_19_0002.abf |              1 |             100    | Slow_I_sin      |
| records/2024_06/06.26/C1/2024_06_26_0003.abf |              1 |              30    | VC r

### Create markdown files

In [7]:
# Sort table by Protocol
table_sorted = sorted(table, key=lambda row: row[3])

# Headers
headers = ["File", "Nb of Sweeps", "Sweep Length (s)", "Protocol"]

# Convert to Markdown
md_table = tabulate(table_sorted, headers=headers, tablefmt="github")

# Save to file
output_file = "abf_table_protocol_sorted.md"
with open(output_file, "w") as f:
    f.write(md_table)

print(f"Markdown table saved to {output_file}")

Markdown table saved to abf_table_protocol_sorted.md


In [8]:
from pathlib import Path
import pyabf
from tabulate import tabulate

records_dir = Path("records")
abf_files = list(records_dir.rglob("*.abf"))

print(f"The directory {records_dir} contains {len(abf_files)} ABF files:")

# Prepare table data
table = []
for file in abf_files:
    abf = pyabf.ABF(file)
    table.append([
        str(file),
        len(abf.sweepList),
        f"{abf.sweepLengthSec:.2f}",
        abf.protocol
    ])

# Headers
headers = ["File", "Nb of Sweeps", "Sweep Length (s)", "Protocol"]

# Generate Markdown table
md_table = tabulate(table, headers=headers, tablefmt="github")

# Write to a Markdown file
output_file = "abf_files_table.md"
with open(output_file, "w") as f:
    f.write(f"# ABF Files Summary\n\n")
    f.write(md_table)

print(f"Markdown table written to {output_file}")

The directory records contains 63 ABF files:
Markdown table written to abf_files_table.md


## Some plots

We use that function:

In [9]:
import sys
sys.path.append("/Users/campillo/Documents/0-git.nosync/data-science-spikes/")  # parent folder containing `utils`
from utils.plots import plot_abf_sweeps_with_legend4
plot_abf_sweeps_with_legend4?

Signature:
plot_abf_sweeps_with_legend4(
    abf,
    file_path=None,
    records_dir=None,
    cmap='tab10',
    lw=0.8,
    alpha=1.0,
    figsize=(10, 6),
    legend_loc='upper right',
    legend_bbox=(1.02, 0.95),
    legend_pad=0.5,
    legend_fontsize=8,
)
Docstring:
Plot all ADC and DAC sweeps with a customizable legend.

Parameters
----------
abf : pyabf.ABF
    Loaded ABF object.
file_path : str, optional
    Path of the ABF file, used to set the figure title.
legend_loc : str
    Reference location of the legend (Matplotlib loc string).
legend_bbox : tuple
    Coordinates to shift legend relative to loc (x, y).
legend_pad : float
    Padding between axes and legend.
legend_fontsize : str or int
    Font size of legend labels.
File:      ~/Documents/0-git.nosync/data-science-spikes/utils/plots.py
Type:      function

We plot all the abf files and put the result in `abf_sweeps_A4.pdf`:

In [11]:
from matplotlib.backends.backend_pdf import PdfPages
import os, io
import sys


records_dir = Path("records")


abf_files = list(records_dir.rglob("*.abf"))

# A4 size (portrait: 8.27 × 11.69 in)
a4_size = (8.27, 11.69)

with PdfPages("abf_sweeps_A4.pdf") as pdf:
    for i in range(0, len(abf_files), 2):
        # master A4 page with 2 slots
        fig, axes = plt.subplots(
            2, 1, figsize=a4_size, constrained_layout=True
        )

        for j in range(2):
            if i + j < len(abf_files):
                file_path = abf_files[i + j]
                abf = pyabf.ABF(file_path)
                print(f"Processing: {file_path}")

                # generate smaller sub-figure
                subfig, (ax1, ax2) = plot_abf_sweeps_with_legend4(
                    abf, file_path=file_path, records_dir=records_dir, figsize=(5, 2.5)  # reduced size
                )

                # save to buffer as image
                buf = io.BytesIO()
                subfig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
                buf.seek(0)
                img = plt.imread(buf)

                # show inside A4 page slot
                axes[j].imshow(img)
                axes[j].axis("off")

                plt.close(subfig)

        pdf.savefig(fig)
        plt.close(fig)

Processing: records/2024_06/06.26/C1/2024_06_26_0025.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0019.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0018.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0024.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0026.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0027.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0023.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0022.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0008.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0020.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0021.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0009.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0010.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0004.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0005.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0011.abf
Processing: records/2024_06/06.26/C1/2024_06_26_0007.abf
Processing: records/2024_06/06.

## Temperature ?

Let's have a look at the methods and attributes of an abf file.

What you’re seeing with `dir(abf)` is a list of all the attributes and methods of the ABF object. Here’s a breakdown:

1. **Attributes starting with `_`**  
   Examples: `_protocolSection`, `_dacSection`, `_dataSection`, `_headerV2`  
   - These are internal structures used by pyabf to store data, DAC signals, protocol info, headers, etc.  
   - They can be accessed, but their structure may not be a simple dictionary.

2. **Attributes without `_`**  
   Examples: `adcNames`, `dacNames`, `sweepX`, `sweepY`, `sweepC`, `fileGUID`, `abfFilePath`  
   - These are the “user-facing” properties. You can use them directly to inspect sweeps, channels, or metadata.

3. **Methods**  
   Examples: `setSweep()`, `getAllXs()`, `getAllYs()`  
   - Functions you can call on the ABF object to manipulate or extract data.

### Example: Inspecting sweeps
```python
import pyabf

abf = pyabf.ABF("records/sample.abf")
print("Sweeps available:", abf.sweepList)      # list of sweep numbers
abf.setSweep(0)                                # set to first sweep
print("X-axis of first sweep:", abf.sweepX)    # x-axis of first sweep
print("ADC data of first sweep:", abf.sweepY)  # ADC data of first sweep
print("DAC data of first sweep:", abf.sweepC)  # DAC data of first sweep
```


### Example: Inspect internal _protocolSection
```python
for attr in dir(abf._protocolSection):
    if not attr.startswith("_"):
        print(attr, getattr(abf._protocolSection, attr))
```

### Example: Searching for temperature-related parameters

```python
def find_temperature(abf):
    # Check protocol section
    for attr in dir(abf._protocolSection):
        if "temp" in attr.lower():
            print("Protocol section:", attr, getattr(abf._protocolSection, attr))
    # Check file comment
    if "temp" in abf.abfFileComment.lower():
        print("File comment:", abf.abfFileComment)
    # Check user list
    if hasattr(abf, "userListParamToVaryName") and abf.userListParamToVaryName:
        for name in abf.userListParamToVaryName:
            if "temp" in name.lower():
                print("User list param:", name)

find_temperature(abf)

```

In [17]:
file_path = "records/2024_06/06.26/C1/2024_06_26_0016.abf"
abf = pyabf.ABF(file_path)

# List all attributes of the protocol section
print(dir(abf._protocolSection))

# Optionally, print attribute names and their values
for attr in dir(abf._protocolSection):
    if not attr.startswith("_"):  # skip internal attributes
        value = getattr(abf._protocolSection, attr)
        print(attr, ":", value)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_blockStart', '_byteStart', '_cleanString', '_entryCount', '_entrySize', '_fb', '_sUnused', 'bEnableFileCompression', 'fADCRange', 'fADCSequenceInterval', 'fAverageWeighting', 'fCellID', 'fDACRange', 'fEpisodeStartToStart', 'fFirstRunDelayS', 'fRunStartToStart', 'fScopeOutputInterval', 'fSecondsPerRun', 'fStatisticsPeriod', 'fSynchTimeUnit', 'fTrialStartToStart', 'fTriggerThreshold', 'lADCResolution', 'lAverageCount', 'lDACResolution', 'lEpisodesPerRun', 'lFileCommentIndex', 'lFinishDisplayNum', 'lNumSamplesPerEpisode', 'lNumberOfTrials', 'lPreTriggerSamples', 'lRunsPerTrial', 'lSamplesPerTrace', 'lStartDisplayNum', 'lStatisticsMe

> NADA : no temperature !